### Dataset

In [1]:
import dspy
from agent.tools.sqlite_tool import get_schema

trainset = [
    dspy.Example(
        question="What is the total revenue of all time?",
        db_schema=get_schema(),
        constraints="Do NOT join with orders table unless filtering by date or customer. All revenue data is in order_items.",
        error_feedback="",
        gold_sql="SELECT SUM(oi.UnitPrice * oi.Quantity * (1 - COALESCE(oi.Discount, 0))) AS TotalRevenue FROM order_items oi",
    ).with_inputs("question", "db_schema", "constraints", "error_feedback"),
    dspy.Example(
        question="How many orders were placed in 1997?",
        db_schema=get_schema(),
        constraints="Use strftime('%Y', OrderDate) for year extraction.",
        error_feedback="",
        gold_sql="SELECT COUNT(*) FROM orders WHERE strftime('%Y', OrderDate) = '1997'",
    ).with_inputs("question", "db_schema", "constraints", "error_feedback"),
    dspy.Example(
        question="Total revenue in 1997",
        db_schema=get_schema(),
        constraints="Join orders only when filtering by date. Revenue = UnitPrice * Quantity * (1 - Discount).",
        error_feedback="",
        gold_sql="SELECT SUM(oi.UnitPrice * oi.Quantity * (1 - COALESCE(oi.Discount, 0))) FROM order_items oi JOIN orders o ON oi.OrderID = o.OrderID WHERE strftime('%Y', o.OrderDate) = '1997'",
    ).with_inputs("question", "db_schema", "constraints", "error_feedback"),
    dspy.Example(
        question="Top 5 best-selling products by total revenue?",
        db_schema=get_schema(),
        constraints="Revenue = UnitPrice * Quantity * (1 - Discount). Show ProductName and revenue.",
        error_feedback="",
        gold_sql="SELECT p.ProductName, SUM(oi.UnitPrice * oi.Quantity * (1 - COALESCE(oi.Discount, 0))) AS Revenue FROM products p JOIN order_items oi ON p.ProductID = oi.ProductID GROUP BY p.ProductID, p.ProductName ORDER BY Revenue DESC LIMIT 5",
    ).with_inputs("question", "db_schema", "constraints", "error_feedback"),
    dspy.Example(
        question="How much has customer SAVEA spent in total?",
        db_schema=get_schema(),
        constraints="Always use CustomerID for matching, not CompanyName. Calculate full revenue with discount.",
        error_feedback="Previous attempts incorrectly used CompanyName in WHERE clause.",
        gold_sql="SELECT SUM(oi.UnitPrice * oi.Quantity * (1 - COALESCE(oi.Discount, 0))) FROM orders o JOIN order_items oi ON o.OrderID = oi.OrderID WHERE o.CustomerID = 'SAVEA'",
    ).with_inputs("question", "db_schema", "constraints", "error_feedback"),
    dspy.Example(
        question="What was the average order value in 1998?",
        db_schema=get_schema(),
        constraints="Calculate per-order total first, then average. Use year from OrderDate.",
        error_feedback="",
        gold_sql="SELECT AVG(OrderTotal) FROM (SELECT o.OrderID, SUM(oi.UnitPrice * oi.Quantity * (1 - COALESCE(oi.Discount, 0))) AS OrderTotal FROM orders o JOIN order_items oi ON o.OrderID = oi.OrderID WHERE strftime('%Y', o.OrderDate) = '1998' GROUP BY o.OrderID)",
    ).with_inputs("question", "db_schema", "constraints", "error_feedback"),
    dspy.Example(
        question="Which customers have never placed an order?",
        db_schema=get_schema(),
        constraints="Use LEFT JOIN from customers to orders and find NULL OrderID.",
        error_feedback="INNER JOIN will exclude these customers.",
        gold_sql="SELECT c.CustomerID, c.CompanyName FROM customers c LEFT JOIN orders o ON c.CustomerID = o.CustomerID WHERE o.CustomerID IS NULL",
    ).with_inputs("question", "db_schema", "constraints", "error_feedback"),
]

testset = [
    dspy.Example(
        question="What are the top 10 most expensive products by unit price?",
        db_schema=get_schema(),
        constraints="Sort by UnitPrice DESC, limit 10.",
        error_feedback="",
        gold_sql="SELECT ProductName, UnitPrice FROM products ORDER BY UnitPrice DESC LIMIT 10",
    ).with_inputs("question", "db_schema", "constraints", "error_feedback"),
    dspy.Example(
        question="How many current and discontinued products are there?",
        db_schema=get_schema(),
        constraints="Use CASE for counting. Assume Discontinued = 1 for yes, 0 for no.",
        error_feedback="",
        gold_sql="SELECT SUM(CASE WHEN Discontinued = 0 THEN 1 ELSE 0 END) AS CurrentProducts, SUM(CASE WHEN Discontinued = 1 THEN 1 ELSE 0 END) AS DiscontinuedProducts FROM products",
    ).with_inputs("question", "db_schema", "constraints", "error_feedback"),
    dspy.Example(
        question="What is the total revenue per customer, including only those from the UK who spent more than 1000?",
        db_schema=get_schema(),
        constraints="Filter by Country='UK', HAVING sum >1000.",
        error_feedback="",
        gold_sql="SELECT c.CompanyName, SUM(oi.UnitPrice * oi.Quantity * (1 - COALESCE(oi.Discount, 0))) AS TotalPaid FROM customers c JOIN orders o ON c.CustomerID = o.CustomerID JOIN order_items oi ON o.OrderID = oi.OrderID WHERE c.Country = 'UK' GROUP BY c.CompanyName HAVING SUM(oi.UnitPrice * oi.Quantity * (1 - COALESCE(oi.Discount, 0))) > 1000",
    ).with_inputs("question", "db_schema", "constraints", "error_feedback"),
    dspy.Example(
        question="For each customer, what is their total revenue and revenue in 1997?",
        db_schema=get_schema(),
        constraints="Use CASE for year-specific sum.",
        error_feedback="",
        gold_sql="SELECT c.CustomerID, c.CompanyName, SUM(oi.UnitPrice * oi.Quantity * (1 - COALESCE(oi.Discount, 0))) AS TotalRevenue, SUM(CASE WHEN strftime('%Y', o.OrderDate) = '1997' THEN oi.UnitPrice * oi.Quantity * (1 - COALESCE(oi.Discount, 0)) ELSE 0 END) AS Revenue1997 FROM customers c JOIN orders o ON c.CustomerID = o.CustomerID JOIN order_items oi ON o.OrderID = oi.OrderID GROUP BY c.CustomerID, c.CompanyName",
    ).with_inputs("question", "db_schema", "constraints", "error_feedback"),
    dspy.Example(
        question="Statistics for products starting with 'Wine': count, total stock, avg price, max/min price ratio, etc.",
        db_schema=get_schema(),
        constraints="Use LIKE 'Wine%'. Round to 2 decimals.",
        error_feedback="",
        gold_sql="SELECT COUNT(*) AS ProductsNumber, SUM(UnitsInStock) AS UnitsNumber, ROUND(AVG(UnitPrice), 2) AS AveragePrice, ROUND(MAX(UnitPrice) / MIN(UnitPrice), 2) AS MaxToMinRatio, ROUND(MAX(UnitPrice) - AVG(UnitPrice), 2) AS MaxToAverage, ROUND(AVG(UnitPrice) - MIN(UnitPrice), 2) AS AverageToMin FROM products WHERE ProductName LIKE 'Wine%'",
    ).with_inputs("question", "db_schema", "constraints", "error_feedback"),
]

### Model

In [2]:
import dspy
from dspy.teleprompt import BootstrapFewShot

MODEL_NAME = "ollama_chat/phi3.5:3.8b-mini-instruct-q4_K_M"
API_BASE = "http://localhost:11434"

lm = dspy.LM(MODEL_NAME, api_base=API_BASE, max_tokens=2048)
dspy.settings.configure(lm=lm)


class TextToSQL(dspy.Signature):
    """Generate executable SQLite query for the Northwind database.
    Be VERY concise in reasoning (1-2 sentences max). Output ONLY the required fields.

    Rules:
    1. Revenue Formula: SUM(UnitPrice * Quantity * (1 - Discount)) from 'order_items'
    2. Dates: SQLite uses strings ('YYYY-MM-DD'). Use strftime('%Y', OrderDate) for years.
    3. Gross Margin: If Cost is missing, use SUM(0.3 * UnitPrice * Quantity * (1 - Discount)) since CostOfGoods ≈ 0.7 * UnitPrice, so margin factor is 0.3.
    4. If needed, map categories via Categories join through products.CategoryID.
    5. Prefer orders + "order_items" + products joins.
    """

    question = dspy.InputField()
    db_schema = dspy.InputField(desc="Table schema with columns")
    constraints = dspy.InputField(desc="Context, date ranges, or KPI formulas from RAG")
    error_feedback = dspy.InputField(
        desc="Error from previous run to fix", optional=True
    )
    gold_sql = dspy.OutputField(desc="A single valid SQLite query string")


class TextToSQLModule(dspy.Module):
    def __init__(self):
        super().__init__()
        self.prog = dspy.ChainOfThought(TextToSQL)

    def forward(self, question, db_schema, constraints, error_feedback=""):
        return self.prog(
            question=question,
            db_schema=db_schema,
            constraints=constraints,
            error_feedback=error_feedback,
        )

In [3]:
from agent.tools.sqlite_tool import run_query


def sql_execution_metric(example, pred, trace=None):
    _, error = run_query(pred.gold_sql)
    if error:
        return 0.0

    df_pred, _ = run_query(pred.gold_sql)
    if df_pred.empty:
        return 0.0

    return 1.0


def evaluate_text_to_sql(module, dataset, label="Generic"):
    print(f"\n--- Evaluating {label} (Text-to-SQL) ---")
    total_score = 0.0
    total = len(dataset)

    for ex in dataset:
        pred = module(
            question=ex.question,
            db_schema=ex.db_schema,
            constraints=ex.constraints,
            error_feedback=ex.get("error_feedback", ""),
        )
        score = sql_execution_metric(ex, pred)
        total_score += score
        print(
            f"Q: {ex.question[:40]}... | Gold SQL: {ex.gold_sql[:40]}... | Pred SQL: {pred.gold_sql[:40]}... | Score: {score:.2f}"
        )

    avg_score = (total_score / total) * 100 if total > 0 else 0
    print(f"Average Score: {avg_score:.2f}%")
    return avg_score

In [ ]:
print(f"Loaded {len(trainset)} examples.")

uncompiled_router = TextToSQLModule()
print("\nRunning Baseline (Uncompiled)...")
score_before = evaluate_text_to_sql(uncompiled_router, testset, label="Baseline")

print("\nRunning Optimization (BootstrapFewShot)...")
teleprompter = BootstrapFewShot(
    metric=sql_execution_metric,
    max_bootstrapped_demos=5,
    max_labeled_demos=3,
    max_rounds=5,
)
with dspy.context(cached=False):
    compiled_router = teleprompter.compile(student=uncompiled_router, trainset=trainset)

print("\nRunning Optimized Model...")
score_after = evaluate_text_to_sql(compiled_router, testset, label="Optimized")

print("\n" + "=" * 30)
print("OPTIMIZATION RESULTS")
print("=" * 30)
print(f"Metric: SQL Accuracy")
print(f"Before: {score_before:.2f}%")
print(f"After:  {score_after:.2f}%")
print(f"Delta:  {score_after - score_before:+.2f}%")

compiled_router.save("optimized_sql.json")
print("\nOptimized router saved to 'optimized_sql.json'")

Loaded 7 examples.

Running Baseline (Uncompiled)...

--- Evaluating Baseline (Text-to-SQL) ---
Q: What are the top 10 most expensive produ... | Gold SQL: SELECT ProductName, UnitPrice FROM produ... | Pred SQL: SELECT ProductName FROM (SELECT p.Produc... | Score: 0.00
Q: How many current and discontinued produc... | Gold SQL: SELECT SUM(CASE WHEN Discontinued = 0 TH... | Pred SQL: SELECT SUM(CASE WHEN p.Discontinued = 0 ... | Score: 1.00
Q: What is the total revenue per customer, ... | Gold SQL: SELECT c.CompanyName, SUM(oi.UnitPrice *... | Pred SQL: SELECT c.CustomerID, SUM(oi.Quantity * o... | Score: 1.00
Q: For each customer, what is their total r... | Gold SQL: SELECT c.CustomerID, c.CompanyName, SUM(... | Pred SQL: SELECT c.CompanyName, SUM(CASE WHEN strf... | Score: 1.00
Q: Statistics for products starting with 'W... | Gold SQL: SELECT COUNT(*) AS ProductsNumber, SUM(U... | Pred SQL: SELECT ROUND(AVG(p.UnitPrice), 2) AS Avg... | Score: 0.00
Average Score: 60.00%

Running Optimiza

 43%|████▎     | 3/7 [00:55<01:14, 18.54s/it]


Bootstrapped 3 full traces after 3 examples for up to 5 rounds, amounting to 3 attempts.

Running Optimized Model...

--- Evaluating Optimized (Text-to-SQL) ---
Q: What are the top 10 most expensive produ... | Gold SQL: SELECT ProductName, UnitPrice FROM produ... | Pred SQL: SELECT p.ProductName, MAX(oi.UnitPrice) ... | Score: 1.00
Q: How many current and discontinued produc... | Gold SQL: SELECT SUM(CASE WHEN Discontinued = 0 TH... | Pred SQL: SELECT SUM(CASE WHEN Discontinued = 0 TH... | Score: 0.00
Q: What is the total revenue per customer, ... | Gold SQL: SELECT c.CompanyName, SUM(oi.UnitPrice *... | Pred SQL: SELECT c.CustomerID, SUM(oi.UnitPrice * ... | Score: 0.00
Q: For each customer, what is their total r... | Gold SQL: SELECT c.CustomerID, c.CompanyName, SUM(... | Pred SQL: SELECT 
    c.CustomerID,
    SUM(oi.Uni... | Score: 0.00
Q: Statistics for products starting with 'W... | Gold SQL: SELECT COUNT(*) AS ProductsNumber, SUM(U... | Pred SQL: SELECT 
    (SELECT COUNT(*) FRO